In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [117]:
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

precipitation_review = (
    project_root / "data" / "raw" / "precipitation" / "precipitation_review.csv"
)
crop_yield_review = (
    project_root / "data" / "raw" / "crop_yield" / "crop_yield_review.csv"
)
vegetation_review = (
    project_root / "data" / "raw" / "vegetation" / "vegetation_review.csv"
)
geography_common = project_root / "data" / "raw" / "geography" / "geography_common.csv"

In [118]:
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

pickle_folder_path = project_root / "data" / "raw" / "pickle"

crop_yield_df = pd.read_pickle(pickle_folder_path / "df_crop_yield_2016_2026.pkl")
geography_df = pd.read_pickle(pickle_folder_path / "df_geography.pkl")
precipitation_df = pd.read_pickle(pickle_folder_path / "df_precipitation_2016_2026.pkl")
vegetation_df = pd.read_pickle(pickle_folder_path / "df_vegetation_2016_2026.pkl")

In [119]:
# renaming the columns

crop_yield_df.rename(
    columns={"state_name": "state", "district_name": "district"}, inplace=True
)
geography_df.rename(columns={"State": "state", "District": "district"}, inplace=True)
precipitation_df.rename(
    columns={"State_Name": "state", "District_Name": "district"}, inplace=True
)
vegetation_df.rename(
    columns={"state_name": "state", "district_name": "district"}, inplace=True
)

In [120]:
# Determining the common state names and removing unwanted records

states_cy = set(crop_yield_df["state"].unique())
states_geo = set(geography_df["state"].unique())
states_per = set(precipitation_df["state"].unique())
states_veg = set(vegetation_df["state"].unique())

common_states = states_cy & states_geo & states_per & states_veg

common_states_df = pd.DataFrame(list(common_states), columns=["state"])
crop_yield_df = crop_yield_df[crop_yield_df["state"].isin(common_states_df["state"])]
geography_df = geography_df[geography_df["state"].isin(common_states_df["state"])]
precipitation_df = precipitation_df[
    precipitation_df["state"].isin(common_states_df["state"])
]
vegetation_df = vegetation_df[vegetation_df["state"].isin(common_states_df["state"])]

In [121]:
# Cleaning up the districts data. Take geography as the base for the canonical dataset

# crop_yield_df['district'].nunique() 608
# geography_df['district'].nunique() 511
# precipitation_df['district'].nunique() 587
# vegetation_df['district'].nunique() 511

In [140]:
from rapidfuzz import process

common_districts_df = geography_df[["state", "district"]].drop_duplicates()

state_districts = common_districts_df.groupby("state")["district"].apply(list).to_dict()


def district_match(row):
    state = row["state"]
    district = row["district"]

    districts_by_state = state_districts.get(state, [])
    match, score, _ = process.extractOne(district, districts_by_state)
    return pd.Series([match, score])


crop_yield_df[["mapped_district", "score"]] = crop_yield_df.apply(
    district_match, axis=1
)
precipitation_df[["mapped_district", "score"]] = precipitation_df.apply(
    district_match, axis=1
)
vegetation_df[["mapped_district", "score"]] = vegetation_df.apply(
    district_match, axis=1
)

In [141]:
common_districts_df.to_csv(geography_common, index=False)
crop_yield_review_df = crop_yield_df[
    ["state", "district", "mapped_district"]
].drop_duplicates()
crop_yield_review_df.to_csv(crop_yield_review, index=False)
precipitation_review_df = precipitation_df[
    ["state", "district", "mapped_district"]
].drop_duplicates()
precipitation_review_df.to_csv(precipitation_review, index=False)
vegetation_review_df = vegetation_df[
    ["state", "district", "mapped_district"]
].drop_duplicates()
vegetation_review_df.to_csv(vegetation_review, index=False)

In [142]:
#  manually reviewed the files and updated *_FINAL.csv files with corrected state district combination

precipitation_district_lookup = (
    project_root / "data" / "raw" / "precipitation" / "precipitation_review_FINAL.csv"
)
crop_yield_district_lookup = (
    project_root / "data" / "raw" / "crop_yield" / "crop_yield_review_FINAL.csv"
)
vegetation_district_lookup = (
    project_root / "data" / "raw" / "vegetation" / "vegetation_review_FINAL.csv"
)

precipatition_district_lookup_df = pd.read_csv(precipitation_district_lookup)
crop_yield_district_lookup_df = pd.read_csv(crop_yield_district_lookup)
vegetation_district_lookup_df = pd.read_csv(vegetation_district_lookup)

In [143]:
vegetation_df.drop(columns=["mapped_district", "score"], inplace=True)
vegetation_df = vegetation_df.merge(
    vegetation_district_lookup_df[["state", "district", "mapped_district"]],
    on=["state", "district"],
    how="left",
)
vegetation_df["district"] = vegetation_df["mapped_district"]
vegetation_df.drop(columns=["mapped_district"], inplace=True)

precipitation_df.drop(columns=["mapped_district", "score"], inplace=True)
precipitation_df = precipitation_df.merge(
    precipatition_district_lookup_df[["state", "district", "mapped_district"]],
    on=["state", "district"],
    how="left",
)
precipitation_df["district"] = precipitation_df["mapped_district"]
precipitation_df.drop(columns=["mapped_district"], inplace=True)

crop_yield_df.drop(columns=["mapped_district", "score"], inplace=True)
crop_yield_df = crop_yield_df.merge(
    crop_yield_district_lookup_df[["state", "district", "mapped_district"]],
    on=["state", "district"],
    how="left",
)
crop_yield_df["district"] = crop_yield_df["mapped_district"]
crop_yield_df.drop(columns=["mapped_district"], inplace=True)

In [152]:
precipitation_processed = project_root / "data" / "processed" / "precipitation.csv"
crop_yield_processed = project_root / "data" / "processed" / "crop_yield.csv"
vegetation_processed = project_root / "data" / "processed" / "vegetation.csv"
geography_processed = project_root / "data" / "processed" / "geography.csv"

vegetation_df.to_csv(vegetation_processed)
precipitation_df.to_csv(precipitation_processed)
crop_yield_df.to_csv(crop_yield_processed)
geography_df.to_csv(geography_processed)


precipitation_processed_pkl = project_root / "data" / "processed" / "precipitation.pkl"
crop_yield_processed_pkl = project_root / "data" / "processed" / "crop_yield.pkl"
vegetation_processed_pkl = project_root / "data" / "processed" / "vegetation.pkl"
geography_processed_pkl = project_root / "data" / "processed" / "geography.pkl"

vegetation_df.to_pickle(vegetation_processed_pkl)
precipitation_df.to_pickle(precipitation_processed_pkl)
crop_yield_df.to_pickle(crop_yield_processed_pkl)
geography_df.to_pickle(geography_processed_pkl)